# Blocks 5 and 6 — the lecture's queries

**The lecture's queries, as Eduardo ran them: for review after class. Nothing here is typed in class.** In the lecture you predicted, with a reason, before a query ran on the projector (*Predict, then watch*), or watched a query run (*Watch*). The headings match the slides, in the same order, numbered.

Block 5 uses the emissions file, `data/raw/env_air_gge_ghg.tsv`. The lab's file is the other one, `env_wasmun.tsv`: Block 5 never touches it.

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 400)   # show every row of a result: a census is never "the top few"

# Anchor to the project folder (DS1, Block 1), then stand there: every path below is from the project folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print("Working in:", Path.cwd())   # must print this project's folder

con = duckdb.connect()             # an in-memory database; it reads the files in data/raw/ directly

### 1. Watch — look before you load (in the terminal, not here)

```bash
head -n 3 data/raw/env_air_gge_ghg.tsv | cut -c 1-90
grep 'CRF1B1,AT' data/raw/env_air_gge_ghg.tsv | cut -c 1-60
```

Three lines, then two: the packed first column, the years as columns, and Austria's `0 d`.

### 2. Predict, then watch — bronze: the file as text. Without `all_varchar`, would DuckDB read the years as numbers or as text?

In [ ]:
con.sql("""
    CREATE OR REPLACE VIEW bronze AS
    SELECT * FROM read_csv('data/raw/env_air_gge_ghg.tsv',
                           delim = '\t', all_varchar = true, names = ['series'])
""")
# The prediction: WITHOUT all_varchar, what type would DuckDB guess for the year columns?
con.sql("""
    SELECT column_name, column_type
    FROM (DESCRIBE SELECT * FROM read_csv('data/raw/env_air_gge_ghg.tsv', delim = '\t'))
    WHERE column_name IN ('2010', '2022', '2024')
""").df()

### 3. Watch — long: split the packed column, turn the years into rows

In [ ]:
con.sql("""
    CREATE OR REPLACE VIEW long AS
    SELECT split_part(series, ',', 1) AS freq,
           split_part(series, ',', 2) AS unit,
           split_part(series, ',', 3) AS airpol,
           split_part(series, ',', 4) AS src_crf,
           split_part(series, ',', 5) AS geo,
           CAST(year AS INTEGER)      AS year,
           cell
    FROM read_csv('data/raw/env_air_gge_ghg.tsv', delim = '\t', all_varchar = true, names = ['series'])
    UNPIVOT INCLUDE NULLS (cell FOR year IN (COLUMNS('^[0-9]{4}$')))
""")
con.sql("SELECT COUNT(*) AS observations FROM long").df()

### 4. Watch — the census of the flag: what follows the space in a cell?

In [ ]:
con.sql("""
    SELECT split_part(cell, ' ', 2) AS flag, COUNT(*) AS n
    FROM long
    GROUP BY flag
    ORDER BY n DESC
""").df()

### 5. Predict, then watch — every cell that is not a number: how many kinds?

In [ ]:
con.sql("""
    SELECT cell, COUNT(*) AS n
    FROM long
    WHERE cell LIKE ':%'
    GROUP BY cell
    ORDER BY n DESC
""").df()

### 6. Predict, then watch — a first silver: no error, and half the rows of long. Did every number survive the cast?

In [ ]:
con.sql("""
    CREATE OR REPLACE VIEW silver_first AS
    SELECT geo, src_crf, year, TRY_CAST(cell AS DOUBLE) AS value
    FROM long
    WHERE unit = 'MIO_T'
""")
con.sql("SELECT COUNT(*) AS rows, COUNT(value) AS with_a_value FROM silver_first").df()

### 7. Watch — what did the cast swallow? Cells where Eurostat wrote a number and the cast gave NULL

In [ ]:
con.sql("""
    SELECT cell, COUNT(*) AS n
    FROM long
    WHERE TRY_CAST(cell AS DOUBLE) IS NULL      -- the cast gave nothing
      AND split_part(cell, ' ', 1) <> ':'       -- but Eurostat wrote a number
    GROUP BY cell
""").df()

### 8. Predict, then watch — zeros, and the naive cast: solid-fuel fugitive emissions (`CRF1B1`), 2022

In [ ]:
con.sql("""
    SELECT COUNT(*) FILTER (WHERE TRY_CAST(cell AS DOUBLE) = 0)                    AS zero_naive_cast,
           COUNT(*) FILTER (WHERE TRY_CAST(split_part(cell, ' ', 1) AS DOUBLE) = 0) AS zero_in_the_file,
           AVG(TRY_CAST(cell AS DOUBLE))                                      AS average_naive_cast,
           AVG(TRY_CAST(NULLIF(split_part(cell, ' ', 1), ':') AS DOUBLE))     AS average_in_the_file
    FROM long
    WHERE unit = 'MIO_T' AND year = 2022 AND src_crf = 'CRF1B1' AND geo <> 'EU27_2020'
""").df()

### 9. Watch — the number, the flag, and the cast; then what the cast swallows now

In [ ]:
con.sql("""
    CREATE OR REPLACE VIEW parsed AS
    SELECT *,
           NULLIF(split_part(trim(cell), ' ', 1), ':') AS number_text,
           NULLIF(split_part(trim(cell), ' ', 2), '')  AS flag
    FROM long
""")
con.sql("""
    SELECT COUNT(*) AS swallowed FROM parsed
    WHERE number_text IS NOT NULL AND TRY_CAST(number_text AS DOUBLE) IS NULL
""").df()

### 10. Predict, then watch — one country, one year, one sector: Germany's total, 2022 (`TOTX4_MEMO`: total excluding LULUCF and memo items)

In [ ]:
con.sql("""
    SELECT unit, TRY_CAST(cell AS DOUBLE) AS value
    FROM long
    WHERE geo = 'DE' AND year = 2022 AND src_crf = 'TOTX4_MEMO'
""").df()

### 11. Watch — the EU among its members, 2022

In [ ]:
con.sql("""
    SELECT SUM(TRY_CAST(cell AS DOUBLE)) FILTER (WHERE geo = 'EU27_2020')                     AS eu27_row,
           SUM(TRY_CAST(cell AS DOUBLE)) FILTER (WHERE geo NOT IN ('EU27_2020', 'CH', 'IS', 'NO', 'TR')) AS sum_of_27_members,
           SUM(TRY_CAST(cell AS DOUBLE))                                                          AS every_row
    FROM long
    WHERE unit = 'MIO_T' AND year = 2022 AND src_crf = 'TOTX4_MEMO'
""").df()

### 12. Predict, then watch — every stage has a named key: is `(geo, year)` the key of `silver_first`?

In [ ]:
con.sql("""
    SELECT COUNT(*)                            AS rows,
           COUNT(DISTINCT (geo, year, src_crf)) AS geo_year_sector,
           COUNT(DISTINCT (geo, year))          AS geo_year
    FROM silver_first
""").df()

### 13. Predict, then watch — the accounting's precedence, as a `CASE`, counted with `FILTER`. Could these counts ever fail to add up?

`CASE WHEN … THEN … END` gives every observation one label: the first `WHEN` that is true wins, so the order of the `WHEN`s is the precedence. `COUNT(*) FILTER (WHERE …)` counts only the rows the condition keeps — one column per destination, and a destination with no rows still shows its 0.

In [ ]:
con.sql("""
    WITH parsed AS (
        SELECT *, NULLIF(split_part(trim(cell), ' ', 1), ':') AS number_text
        FROM long
    ),
    sorted AS (
        SELECT *,
               CASE
                   WHEN cell IS NULL OR trim(cell) = ''         THEN 'rejected'    -- empty: not even a ':'
                   WHEN number_text IS NOT NULL
                        AND TRY_CAST(number_text AS DOUBLE) IS NULL THEN 'rejected'    -- a number that does not parse
                   WHEN unit <> 'MIO_T'                         THEN 'excluded'    -- not silver's unit
                   WHEN geo = 'EU27_2020'                       THEN 'aggregate'   -- kept, marked
                   ELSE 'retained'                                                 -- a ':' lands here, as NULL
               END AS destination
        FROM parsed
    )
    SELECT COUNT(*)                                          AS observations_in,
           COUNT(*) FILTER (WHERE destination = 'rejected')  AS rejected,
           COUNT(*) FILTER (WHERE destination = 'excluded')  AS excluded,
           COUNT(*) FILTER (WHERE destination = 'aggregate') AS aggregate,
           COUNT(*) FILTER (WHERE destination = 'retained')  AS retained,
           COUNT(*) FILTER (WHERE destination = 'retained' AND number_text IS NULL) AS retained_not_available
    FROM sorted
""").df()

---

# Block 6 — checks on the Lab 5 silver

These queries read `data/silver/waste.parquet`, the silver this folder's pipeline writes. In class they ran on a finished Lab 5 (4,828 values). In the Lab 5 starter as cloned, they show the colleague's silver (4,022 values) until `scripts/clean.py` is fixed.

### 14. First: run this folder's pipeline, so the silver file exists

In [ ]:
# The Block 6 queries read the silver file this folder's pipeline writes. Run the pipeline once, so it exists.
import subprocess
import sys

subprocess.run([sys.executable, "pipeline.py"], check=True, capture_output=True)
print("pipeline ran: data/silver/waste.parquet is this folder's silver")

### 15. Watch — the key check, on the Lab 5 silver

In [ ]:
con.sql("""
    SELECT COUNT(*)                             AS n_rows,
           COUNT(DISTINCT (geo, year, wst_oper)) AS n_keys,
           COUNT(DISTINCT (geo, year))           AS n_geo_year
    FROM 'data/silver/waste.parquet'
""").df()

### 16. Predict, then watch — a key check that cannot see a duplicate. Does it pass? What wrong silver would it also pass?

In [ ]:
con.sql("""
    SELECT COUNT(*) AS rows_with_no_geo
    FROM 'data/silver/waste.parquet'
    WHERE geo IS NULL
""").df()

### 17. Predict, then watch — rows, and values: the colleague's silver against a finished one

In [ ]:
con.sql("""
    SELECT COUNT(*) AS n_rows, COUNT(value) AS n_values
    FROM 'data/silver/waste.parquet'
""").df()

### 18. The demo, as it ran: the Lab 5 pipeline gains three checks

Run on the projector in a throwaway copy of a finished Lab 5 (never your own `data/raw/`), with a `scripts/checks.py` holding three checks — key on silver, domain on values and flags, count of raw `KG_HAB` observations against silver + rejects — added to `pipeline.py`.

1. Clean file: `3 checks, none says stop`.
2. One real raw row (`A,GEN,KG_HAB,DE`) appended a second time: `STOP: silver_key observed 15, wanted 0 duplicate (geo, year, wst_oper)`. One raw row is fifteen observations, one per year.
3. The key check taken out of `CHECKS`: `2 checks, none says stop` — and gold's row for Germany reads 2010: 1,204, 2024: 1,256 (flag `e`), change 52. The file says 628 for 2024: `report.py` summed two copies. The count check passed, because the duplicate is in the raw file and in silver alike.

---

# Self-study

## What to remember from Block 5

- **Bronze, silver, gold.** Bronze is the file as it arrived, never edited. Silver is typed, one unit, one row per a key
  you can state, with flags kept and a data dictionary. Gold is the answer at the question's grain. Each layer is made
  by code from the one below, and running the code twice makes the same tables.
- **Look before you load:** `head` in the terminal, then read every cell as text (`all_varchar = true`), so nothing is
  converted before the step your script controls.
- **Reshape:** `split_part` unpacks a packed column; `UNPIVOT INCLUDE NULLS` turns year columns into rows without
  dropping an empty cell. Say the count before you run it: rows × year columns.
- **The census first:** `GROUP BY` the column and read every value, before you convert anything.
- **Missing is not one thing:** `:` means "no number here", and the letter after it says why. `NULLIF(x, ':')` turns
  the marker into `NULL` on purpose. Zero is not missing.
- **A cast that returns `NULL` instead of an error swallows values in silence** (`TRY_CAST`, and pandas'
  `to_numeric(errors="coerce")`). Count what it swallowed: the cells where the cast gave nothing and the source gave
  something.
- **Keep the flag** in its own column: it is information about the number next to it.
- **Units are a filter**, and silver states its unit. **Aggregates sit among members:** keep them, marked.
- **Every stage has a key.** A key check is written against the key of the stage it checks.
- **The accounting:** every observation lands in exactly one place, decided in order (rejected → excluded → aggregate →
  retained), and the counts add up to the observations in. Count what was written, not your own labels.

## What to remember from Block 6

- **A check that cannot fail is not a check.** A validation is an assertion that stops the run, and its message
  carries the number.
- **Four families:** key (unique, never `NULL`, on the stage's own key); domain (allowed values from a census, ranges,
  one unit, dates inside the window); count (rows in = rows out + rejected + excluded; a join keeps the left count;
  `GROUP BY` hides a missing period); reconciliation (two independent computations agree, by the identity the metric
  permits: sums for additive measures, numerator and denominator for a ratio, weights for an average).
- **Compare sums with a tolerance**, never `==`: float and money sums differ in the last digits, and sources round.
  `abs(a - b) <= tolerance`, or `ROUND` both sides, and put the difference in the message.
- **For every check, name the wrong data it would pass.** Incorrect, weak, tautological, missing.
- **A row you reject and a run you stop:** a bad row is quarantined with a reason and counted; a broken invariant
  (a duplicate key, a sum that does not match, rows that vanished) stops the run.
- **Idempotency** is a check: run twice, compare the files.
- **The fresh-clone test:** clone, `uv sync`, run. Nothing may depend on your laptop.
- **The note:** the metric (grain, filter, unit), what was excluded and how many, the assumptions, the reconciliation,
  and what the data cannot answer — as a claim about the data, not a hedge.

## Common mistakes

- Casting the whole cell: `TRY_CAST('0 d' AS DOUBLE)` is `NULL`, and nothing says so.
- `COALESCE(value, 0)` to "fill the gaps": it turns "we don't know" into "nothing", the opposite mistake.
- Throwing the flag away once the number parses.
- Summing a column without filtering its unit: in the emissions file, every row is there in two units, and the total
  comes out 1,001 times too big.
- Summing a table that holds a group of countries beside its members.
- Checking silver for the gold table's key, or checking uniqueness with `IS NOT NULL`.
- An accounting that counts the labels it just assigned: it always adds up, so it cannot fail.
- `assert n > 0`, `assert total == total`, or a `try`/`except` around an assert: checks that pass on wrong data.
- Comparing money or float sums with `==`.
- A row count check on a step that changes values, not rows: the rows are all there; the values are not.
- A note that says "more research is needed" instead of what the data has no column for.